# More processing data with pandas 

Along the way we will cover a number of useful techniques in pandas including:

- renaming columns
- iterating data frame rows and applying functions
- data aggregation
- repeating the analysis task for several input files

Sources  : https://geo-python-site.readthedocs.io/en/latest/ running on a Attribution-ShareAlike 4.0 International Licence https://creativecommons.org/licenses/by-sa/4.0/deed.en 

## Input data

In the lesson this week we are using **simulated** weather data for five Swedish cities: Stockholm, Gothenburg, Malmö, Uppsala and Västerås. Each file contains four observations per day (at 00, 06, 12 and 18 UTC) from 1 January 2018 to 31 December 2020, with temperatures in degrees Fahrenheit. The values were generated by a computer program, but they follow the Swedish climate of 2018-2020. The file `dataset/README.md` describes all columns and their units.

## Reading the data

In order to get started, let's first import pandas: 

In [ ]:
# Import pandas, as always, with the short nickname pd
import pandas as pd

At this point, we can already have a quick look at the data file `Stockholm_continuous_final.txt` and how it is structured. We can notice at least two things we need to consider when reading in the data:

> **Input data structure**
>
> - **Delimiter:** The columns are separated by **whitespace** (tab characters in these files; other files may use a varying number of spaces). If you check out the documentation for the [read_csv() method](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html), you can see that there are two different ways of handling this: `sep=r'\s+'` or `delim_whitespace=True` (but never both at the same time). In this course we use `sep=r'\s+'`, because the `delim_whitespace` parameter is deprecated in recent versions of pandas and will eventually be removed.
>
> - **No Data values:** No data values in the simulated data are coded with a varying number of `*` characters. We can tell pandas to treat those as NaNs by specifying `na_values=['*', '**', '***', '****', '*****', '******']`.

In [ ]:
# The data file lives in a 'dataset' folder next to this notebook.
# This is a RELATIVE path, so it works on any computer as long as the
# folder sits in the same place relative to the notebook.
fp = "dataset/Stockholm_continuous_final.txt"

In [ ]:
# Read the data file.
# sep=r'\s+' means "the columns are separated by one or more spaces".
# na_values tells pandas to treat the runs of * characters as missing (NaN) values.
data = pd.read_csv(
    fp, sep=r'\s+', na_values=["*", "**", "***", "****", "*****", "******"]
)


In [ ]:
# Check the first rows to see that the file was read correctly
data.head()

All seems ok. However, we won't be needing all of the 8 columns for detecting warm temperatures in July. We can check all column names by running `data.columns`:

In [ ]:
# List all the column names that were read from the file
data.columns


### Reading in the data once again

This time, we will read in only some of the columns using the `usecols` parameter. Let's read in the columns that are useful for our analysis: the timestamp, the temperature, the precipitation, and the station location - that is, `'YRMODAHRMN'`, `'Temperature'`, `'Precipitation'`, and `'City'`.

In [ ]:
# Read in only the selected columns, using usecols
data = pd.read_csv(
    fp,
    sep=r'\s+',
    usecols=["YRMODAHRMN","Temperature", "Precipitation","City"],
    na_values=["*", "**", "***", "****", "*****", "******"],
)

# Check the dataframe
data.head()

Okay so we can see that the data was successfully read to the DataFrame and we also seemed to be able to convert the asterisk (\*) characters into `NaN` values. 

## Renaming columns

Let's rename the columns  using the [rename](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html) function. In order to change the column names, we need to tell pandas how we want to rename the columns using a dictionary that lists old and new column names

Let's first check again the current column names in our DataFrame:

In [ ]:
# Check the current column names before we rename them
data.columns

We can define the new column names using a [dictionary](https://docs.python.org/3/tutorial/datastructures.html#dictionaries) where we list `key: value` pairs, in which the original column name (the one which will be replaced) is the key and the new column name is the value.

> **Dictionaries**
>
> A [dictionary](https://docs.python.org/3/tutorial/datastructures.html#dictionaries) is a specific type of data structure in Python for storing key-value pairs. In this course, we will use dictionaries mainly when renaming columns in a pandas DataFrame, but dictionaries are useful for many different purposes! For more information about Python dictionaries, check out [this tutorial](https://realpython.com/python-dicts/).

Let's change the following:

- `Temperature` to `Temp_Average`
- `Precipitation` to `PRECIP`
- `City` to `Location`
- `YRMODAHRMN` to `Date`



In [ ]:
# Create a dictionary of old name: new name pairs
new_names = {"Temperature": "Temp_Average", "Precipitation": "PRECIP", "City": "Location","YRMODAHRMN": "Date"}

# Let's see what the variable new_names looks like
new_names

In [ ]:
# Check the data type of the new_names variable - it is a dictionary
type(new_names)

From above we can see that we have successfully created a new dictionary. 

Now we can change the column names by passing that dictionary using the parameter `columns` in the `rename()` function:

In [ ]:
# Rename the columns
data = data.rename(columns=new_names)

# Print the new columns
print(data.columns)

Perfect, now our column names are easier to understand and use. 

### Check your understanding

The temperature values in our data files are again in Fahrenheit. As you might guess, we will soon convert these temperatures into Celsius. In order to avoid confusion between the columns, let's rename the column `Temp_Average` to `TEMP_F`. 

In [ ]:
# Solution
# Create the dictionary with old and new names
new_names = { "Temp_Average": "TEMP_F"}

# Rename the columns
data = data.rename(columns=new_names)

# Check the output
data.head()

## Data properties

As we learned last week, it's always a good idea to check basic properties of the input data before proceeding with the data analysis. Let's check the:

- Number of rows and columns

In [ ]:
# Number of (rows, columns)
data.shape

- Top and bottom rows

In [ ]:
# The first five rows
data.head()

In [ ]:
# The last five rows
data.tail()

- Data types of the columns

In [ ]:
# The data type of each column
data.dtypes

- Descriptive statistics

In [ ]:
# Descriptive statistics for the numeric columns
data.describe()

Here we can see that there are varying number of observations per column (look at the `count` row above), because some of the columns have missing values.

## Using your own functions in pandas 

Now it's again time to convert temperatures from Fahrenheit to Celsius! Yes, we have already done this many times before, but this time we will learn how to apply our own functions to data in a pandas DataFrame.

First, we will define a function for the temperature conversion, and then we will apply this function for each Fahrenheit value on each row of the DataFrame. The output celsius values will be stored in a new column called `TEMP_C`.

To begin we will see how we can apply the function row-by-row using a `for` loop and then we will learn how to apply the function to all rows more efficiently all at once.

### Defining the function

For both of these approaches, we first need to define our function to convert temperature from Fahrenheit to Celsius.

In [ ]:
def fahr_to_celsius(temp_fahrenheit):
    """Function to convert Fahrenheit temperature into Celsius.

    Parameters
    ----------

    temp_fahrenheit: int | float
        Input temperature in Fahrenheit (should be a number)

    Returns
    -------

    Temperature in Celsius (float)
    """

    # Convert the Fahrenheit into Celsius
    converted_temp = (temp_fahrenheit - 32) / 1.8

    return converted_temp

To make sure everything is working properly, let's test the function with a known value.

In [ ]:
# Test the function with a known value: 32°F should come out as exactly 0°C
fahr_to_celsius(32)

Let's also print out the first rows of our data frame to see our input data before further processing.

In [ ]:
# Look at the data before we start converting anything
data.head()

### Iterating over rows

We can use the function one row at a time using a `for` loop and the [iterrows()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.iterrows.html) method. This will allow us to iterate row by row using `iterrows()` in a `for` loop to repeat a given process *for each row in a pandas DataFrame*. Please note that iterating over rows is a rather inefficient approach, but it is still useful to understand the logic behind the iteration.

When using the `iterrows()` method it is important to understand that `iterrows()` accesses not only the values of one row, but also the `index` of the row as well. 

Let's start with a simple for loop that goes through each row in our DataFrame.

> **Note:**
>
> We use single quotes to select the column `TEMP_F` of the row in the example below. In Python versions before 3.12, using double quotes inside a double-quoted f-string causes a `SyntaxError`, because Python interprets the inner quote as the end of the string. Mixing the two kinds of quotes works in every version.

In [ ]:
# Loop over the rows. iterrows() gives us TWO things each time round:
# the row's index (idx) and the row's data (row, which is a pandas Series).
for idx, row in data.iterrows():

    # Print the index value
    print(f"Index: {idx}")

    # Print the row
    print(f"Temp F: {row['TEMP_F']}\n")

    # break stops the loop after the first row, so we don't print thousands of lines
    break

> **Breaking a loop**
>
> When developing code in a `for` loop, you do not always need to go through the entire loop in order to test things out. 
> The [break](https://docs.python.org/3/reference/simple_stmts.html#break) statement in Python terminates the current loop wherever it is placed, and we can use it here just to check the values on the first row (based on the first iteration in the `for` loop).
> This can be helpful when working with a large data file or dataset, because you might not want to print thousands of values to the screen!
> For more information, check out [this tutorial](https://www.tutorialspoint.com/python/python_break_statement.htm).

We can see that the `idx` variable indeed contains the index value at position 0 (the first row) and the `row` variable contains all the data from that given row stored as a pandas `Series`.

Let's now create an empty column `TEMP_C` for the Celsius temperatures and update the values in that column using the `fahr_to_celsius` function we defined earlier.

In [ ]:
# Create an empty float column to hold the output values
data["TEMP_C"] = 0.0

# Iterate over the rows
for idx, row in data.iterrows():

    # Convert the Fahrenheit value on this row into Celsius
    celsius = fahr_to_celsius(row["TEMP_F"])

    # Write the converted value back into the DataFrame at this row's index.
    # Note that we update 'data' here, not 'row' - see the pitfall below.
    data.at[idx, "TEMP_C"] = celsius

> ⚠️ **Common Pitfall: changing `row` inside `iterrows()` does nothing**
>
> `iterrows()` hands you a **copy** of each row, not the row itself. So this looks correct but silently changes nothing at all:
>
> ```python
> for idx, row in data.iterrows():
>     row["TEMP_C"] = fahr_to_celsius(row["TEMP_F"])   # WRONG - the DataFrame is unchanged
> ```
>
> To actually update the DataFrame you must write back into it using the row's index, as we do above:
>
> ```python
> data.at[idx, "TEMP_C"] = fahr_to_celsius(row["TEMP_F"])   # correct
> ```

> **Reminder: .at or .loc?**
>
> Here, you could also use `data.loc[idx, new_column] = celsius` to achieve the same result. 
>
> If you only need to access a single value in a DataFrame, [DataFrame.at](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.at.html) is faster than [DataFrame.loc](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.loc.html), which is designed for accessing groups of rows and columns. 

Finally, let's see how our DataFrame looks like now after the calculations above.

In [ ]:
# The new TEMP_C column now holds the converted values
data.head()

### Applying the function

pandas DataFrames and Series have a dedicated method `.apply()` for applying functions on columns (or rows!). When using `.apply()`, we pass the function name (without parentheses!) as an argument to the `apply()` method. Let's start by applying the function to the `TEMP_F` column that contains the temperature values in Fahrenheit.

In [ ]:
# apply() runs our function on every value in the column.
# Note that we pass the function NAME, with no parentheses after it.
data["TEMP_F"].apply(fahr_to_celsius)

> ⚠️ **Common Pitfall: pass the function name to `.apply()`, without brackets**
>
> When using `.apply()`, you give it the *name* of the function so that pandas can call it for you, once per value:
>
> ```python
> data["TEMP_F"].apply(fahr_to_celsius)     # correct
> data["TEMP_F"].apply(fahr_to_celsius())   # WRONG - raises a TypeError
> ```
>
> Adding brackets means "call this function right now, with no arguments", which fails immediately. Think of it as handing pandas the *recipe*, not the finished dish.

The results look logical, so we can store them permanently into the `TEMP_C` column (overwriting the old values).

In [ ]:
# Store the converted values permanently in the TEMP_C column
data["TEMP_C"] = data["TEMP_F"].apply(fahr_to_celsius)

If a DataFrame has several temperature columns, you can convert them all at once by selecting them first and then calling `.apply()`, for example `data[["TMAX_F", "TMIN_F"]].apply(fahr_to_celsius)`. Our data only has one temperature column, so we do not need this here.

> **Note:**
>
> Applying the function on *all* columns using `data.apply(fahr_to_celsius)` does not work here: it raises a `TypeError`, because the `Location` column contains text. And even for numeric columns, converting values that are not Fahrenheit temperatures (such as precipitation or dates) would not make sense.

You might also notice that our conversion function would also allow us to 
pass one column or the entire dataframe as a parameter. For example, like this: `fahr_to_celsius(data["TEMP_F"])`. However, the code is perhaps easier to follow when using the apply method.

Let's now take a look at the DataFrame contents.

In [ ]:
# Look at the first 10 rows, to see both temperature columns side by side
data.head(10)

> **Should I use .iterrows() or .apply()?**
>
> We are teaching the `.iterrows()` method because it helps to understand the structure of a DataFrame and the process of looping through DataFrame rows. However, using `.apply()` is often more efficient in terms of execution time. 
>
> At this point, the most important thing is that you understand what happens when you are modifying the values in a pandas DataFrame. When doing the course exercises, either of these approaches is ok!

## Parsing dates

As part of this lesson, we eventually want to group our data based on month in order to see whether the temperatures in July of 2018 were warmer than average. Currently, the date and time information is stored in the column `Date` (which was originally titled `YRMODAHRMN`):

`YRMODAHRMN = YEAR-MONTH-DAY-HOUR-MINUTE IN GREENWICH MEAN TIME (GMT)`

Let's have a closer look at the date and time information we have by checking the values in that column, and their data type.

In [ ]:
# The first 10 timestamps
data["Date"].head(10)

In [ ]:
# The last 10 timestamps
data["Date"].tail(10)

The `Date` column contains four observations per day, at 00, 06, 12 and 18 UTC. The timestamp for the first observation is `201801010000`, i.e. from the 1st of January 2018 at 00:00, and the timestamp for the latest observation is `202012311800`, from the 31st of December 2020 at 18:00.

In [ ]:
# What data type are the timestamps stored as?
data["Date"].dtypes

The information in this column is stored as integer values.

We now want to aggregate the data on a monthly level, and in order to do so we need to "label" each row of data based on the month when the record was observed. In order to do this, we need to somehow separate information about the year and month for each row.

We can create these "labels" by making a new column containing information about the month (including the year, but excluding day, hours, and minutes).

Before further taking that step, we should first convert the contents in the `Date` column to a new column with character string values for convenience.

In [ ]:
# Convert the whole timestamp column into text (strings),
# so that we can cut pieces out of it.
data["Date_STR"] = data["Date"].astype(str)

### String slicing

Now that we have converted the date and time information into character strings, we next need to "cut" the needed information from the [string objects](https://docs.python.org/3/tutorial/introduction.html#strings). If we look at the latest time stamp in the data (`202012311800`), you can see that there is a systematic pattern `YEAR-MONTH-DAY-HOUR-MINUTE`. Four first characters represent the year, and six first characters are year and month!

In [ ]:
# A single example timestamp.
# Characters 0 to 5 give the year and month together ("202012").
date = "202012311800" 
date[0:6]

Based on this information, we can slice the correct range of characters from the `Date_STR` column using the [pandas.Series.str.slice()](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.slice.html) method.

In [ ]:
# Take characters 0-5 of every timestamp: the year and month, e.g. "201801"
data["YEAR_MONTH"] = data["Date_STR"].str.slice(start=0, stop=6)

# Let's see what we have
data.head()

> ⚠️ **Common Pitfall: the `stop` value is not included when slicing**
>
> Just like list indexing and `range()`, string slicing **stops before** the `stop` position. Using the timestamp `201801010000`:
>
> | Slice | Characters taken | Result |
> | --- | --- | --- |
> | `start=0, stop=4` | 0, 1, 2, 3 | `"2018"` (the year) |
> | `start=4, stop=6` | 4, 5 | `"01"` (the month) |
> | `start=0, stop=6` | 0 to 5 | `"201801"` (year + month) |
>
> A very easy mistake is writing `start=5, stop=6`, which takes only **one** character. That happens to look correct for months 1-9, but returns `"0"` for October, November, and December - so always count the positions carefully.

Nice! Now we have "labeled" the rows based on information about date and time, but only including the year and month in the labels.

### Check your understanding

Create a new column called `'MONTH'` with information about the month of each observation, but excluding the year.

In [ ]:
# Solution
# Characters 4-5 of the Date_STR column are the month on its own, e.g. "01"
data["MONTH"] = data["Date_STR"].str.slice(start=4, stop=6)

# Check the result
data[["YEAR_MONTH", "MONTH"]]

## Aggregating data in pandas by grouping

Here, we will learn how to use [pandas.DataFrame.groupby](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html), which is a handy method for combining large amounts of data and computing statistics for subgroups.

In our case, we will use the groupby method to calculate the average temperatures for each month through these three steps:

  1. Grouping the data based on the year and month
  2. Calculating the average for each month (each group) 
  3. Storing those values into a new DataFrame called `monthly_data`

Before we start grouping the data, let's once again see what our data looks like.

In [ ]:
# How many rows do we have in total?
print(f"number of rows: {len(data)}")

In [ ]:
# A reminder of what the data looks like
data.head()

We have quite a few rows of weather data. Our goal is to create an aggregated data frame that has only one row per month.

To condense our data to monthly average values we can group our data based on the unique year and month combinations.

In [ ]:
# Group the rows by their YEAR_MONTH label.
# This does not calculate anything yet - it just sorts the rows into groups.
grouped = data.groupby("YEAR_MONTH")

> **Note:**
>
> It is also possible to create combinations of years and months on-the-fly when grouping the data (if the data had separate `YEAR` and `MONTH` columns - we create such columns at the end of this lesson):
>
> ```python
> # Group the data 
> grouped = data.groupby(['YEAR', 'MONTH'])
> ```

Now, let's explore the new variable `grouped`.

In [ ]:
# What type is the grouped object?
type(grouped)

In [ ]:
# How many groups are there? (one for each unique YEAR_MONTH)
len(grouped)

We have a new object with type `DataFrameGroupBy` with 36 groups. In order to understand what just happened, let's also check the number of unique year and month combinations in our data.

In [ ]:
# The number of unique year-month labels should match the number of groups
data["YEAR_MONTH"].nunique()

Length of the grouped object should be the same as the number of unique values in the column we used for grouping. For each unique value, there is a group of data.

Let's explore our grouped data even further. 

We can check the "names" of each group.

In [ ]:
# This prints the "key" (the YEAR_MONTH label) of every group.
# With three years of data there are 36 of them.
grouped.groups.keys()

### Accessing data for one group

Let us now check the contents of the group representing January 2018 (the name of that group is `201801`). We can get the values for that month from the grouped object using the `get_group()` method.

In [ ]:
# Pick one month, written as a character string to match the YEAR_MONTH labels
month = "201801"

# Select just that group
group1 = grouped.get_group(month)

In [ ]:
# Let's see what a single group contains
group1

Ahaa! As we can see, a single group contains a DataFrame with values only for that specific month and year. Let's check the DataType of this group.

In [ ]:
# A single group is itself an ordinary pandas DataFrame
type(group1)

So, as noted above, one group is a pandas DataFrame! This is really useful, because we can now use all the familiar DataFrame methods for calculating statistics, etc. for this specific group. We can, for example, calculate the average values for all variables using the statistical functions that we have seen already (e.g. mean, std, min, max, median, etc.).

We can do that by using the `mean()` function that we already used during Lesson 5. Let's calculate the mean for the following attributes all at once:

- `PRECIP`
- `TEMP_F`
- `TEMP_C`


In [ ]:
# Specify the columns that will be part of the calculation
mean_cols = ["PRECIP", "TEMP_F", "TEMP_C"]

# Calculate the mean values all at one go
mean_values = group1[mean_cols].mean()

# Let's see what we have
print(mean_values)

Above, we saw how you can access data from a single group. In order to get information about all groups (all months) we can use a `for` loop or methods available in the grouped object.

### For loops and grouped objects

When iterating over the groups in our `DataFrameGroupBy` object it is important to understand that a single group in our `DataFrameGroupBy` actually contains not only the actual values, but also information about the `key` that was used to do the grouping. Hence, when iterating over the data we need to assign the `key` and the values into separate variables.

So, let's see how we can iterate over the groups and print the key and the data from a single group (again using `break` to only see what is happening for the first group).

In [ ]:
# Iterate over the groups.
# Each time round, 'key' is the group's YEAR_MONTH label and 'group' is its DataFrame.
for key, group in grouped:
    # Print key and group
    print(f"Key:\n {key}")
    print(f"\nFirst rows of data in this group:\n {group.head()}")

    # Stop iteration with break command
    break

OK, so from here we can see that the `key` contains the name of the group `YEARMONTH`.

Let's build on this and see how we can create a DataFrame where we calculate the mean values for all those weather attributes that we were interested in. We will repeat some of the earlier steps here so you can see and better understand what is happening.

In [ ]:
# Create an empty DataFrame for the aggregated values
monthly_data = pd.DataFrame()

# The columns that we want to aggregate
mean_cols = ["PRECIP", "TEMP_F", "TEMP_C"]

# Iterate over the groups
for key, group in grouped:

    # Calculate mean
    mean_values = group[mean_cols].mean()

    # Add the ´key´ (i.e. the date+time information) into the aggregated values
    mean_values["YEAR_MONTH"] = key
    
    # Convert the mean_values series to a DataFrame and make it have a row orientation
    row = mean_values.to_frame().transpose()

    # Concatenate the aggregated values into the monthly_data DataFrame
    monthly_data = pd.concat([monthly_data, row], ignore_index=True)

> **Note:**
>
> You can find a slightly more efficient approach for this same kind of aggregation in [Chapter 3 of the *Python for Geographic Data Analysis*](https://python-gis-book.readthedocs.io/en/latest/part1/chapter-03/nb/02-data-analysis.html#aggregating-data-with-groupby) textbook.

Now, let us see what we have.

In [ ]:
# Show the aggregated monthly values
print(monthly_data)

> **Tip:** pandas can do the same aggregation in a single line. As a bonus, the result keeps proper numeric data types (in the loop above, adding the text label to each row turns every column into the generic `object` type).

In [ ]:
# The same monthly means in one line, using groupby() directly
monthly_data_short = data.groupby("YEAR_MONTH")[mean_cols].mean().reset_index()
monthly_data_short.head()

Awesome! Now we have aggregated our data and we have a new DataFrame called `monthly_data` where we have mean values for each month in the data set.

## Detecting warm months

Now that we have aggregated our data on monthly level, all we need to do is to sort our results in order to check which years had the warmest July temperatures. A simple approach is to select all Julys from the data, group the data and check which group(s) have the highest mean value.

We can start this by selecting all records that are from July (regardless of the year).

In [ ]:
# A reminder of the full data set
data.head()

In [ ]:
# Keep only the rows where the MONTH label is "07" (July), from any year
Julys = data[data["MONTH"] == "07"]

Next, we can take a subset of columns that might contain interesting information.

In [ ]:
# Keep only the columns we are interested in
Julys = Julys[[ "TEMP_F", "TEMP_C", "YEAR_MONTH"]]

We can group by year and month.

In [ ]:
# Group the July rows by their year-and-month label
grouped = Julys.groupby(by="YEAR_MONTH")

And then we can calculate the mean for each group.

In [ ]:
# Calculate the mean of each group
monthly_mean = grouped.mean()

In [ ]:
# The mean July values for each year
monthly_mean.head()

Finally, we can sort and check the highest temperature values. We can sort the data frame in a descending order to do this.

In [ ]:
# Sort from warmest to coldest: ascending=False puts the HIGHEST values first
monthly_mean.sort_values(by="TEMP_C", ascending=False).head()

So, how did July 2018 rank at the Stockholm observation station? 

## Repeating the data analysis with a larger dataset

To wrap up today's lesson, let's repeat the data analysis steps above for all the available data we have (!). First, it would be good to confirm the path to the directory where all the input data are located.

The idea is, that we will repeat the analysis process for each input file using a (rather long) for loop! Here we have all the main analysis steps with some additional output info, all in one long code cell.

In [ ]:
# Read selected columns of data, using whitespace (spaces or tabs) as the separator
# and treating the * characters as NoData values
data = pd.read_csv(
    fp,
    sep=r'\s+',
    usecols=["YRMODAHRMN","Temperature", "Precipitation","City"],
    na_values=["*", "**", "***", "****", "*****", "******"],
)

# Rename the columns
new_names = {"Temperature": "Temp_Average_F", "Precipitation": "PRECIP", "City": "Location","YRMODAHRMN": "Date"}

data = data.rename(columns=new_names)

# Print info about the current input file:
print(f"Location: {data.at[0, 'Location']}")
print(f"NUMBER OF OBSERVATIONS: {len(data)}")

# Create the column that will hold the Celsius values
col_name = "TEMP_C"
data[col_name] = None

# Convert temperatures from Fahrenheit to Celsius
data["TEMP_C"] = data["Temp_Average_F"].apply(fahr_to_celsius)

# Convert the timestamp to a string so that we can slice it
data["Date_STR"] = data["Date"].astype(str)

# Parse year and month out of the timestamp.
# Characters 0-3 are the year and characters 4-5 are the month.
data["MONTH"] = data["Date_STR"].str.slice(start=4, stop=6).astype(int)
data["YEAR"] = data["Date_STR"].str.slice(start=0, stop=4).astype(int)

# Extract observations for the month of July
Julys = data[data["MONTH"] == 7]

# Take a subset of columns
Julys = Julys[[ "Temp_Average_F", "TEMP_C", "YEAR", "MONTH"]]

# Group by year and month
grouped = Julys.groupby(by=["YEAR", "MONTH"])

# Get mean values for each group
monthly_mean = grouped.mean()

# Print the five warmest Julys
print(monthly_mean.sort_values(by="TEMP_C", ascending=False).head(5))
print("\n")

At this point we will use the `glob()` function from the module `glob` to list our input files. glob is a handy function for finding files in a directory that match a given pattern, for example.

In [ ]:
# The glob module helps us find all the files matching a given pattern
import glob

In [ ]:
# Find every .txt file inside the 'dataset' folder.
# Adjust this path if your data lives somewhere else.
# sorted() puts the file names in alphabetical order (glob returns them in no particular order)
file_list = sorted(glob.glob("dataset/*txt"))

> **Note:**
>
> Note that we're using the `*` character as a wildcard, so **any** file inside the `dataset` folder whose name ends with `txt` will be added to the list of files we iterate over. If that folder also contains metadata files ending in `.txt`, you would need a more specific pattern to leave them out.

In [ ]:
# How many files did we find, and what are they called?
print(f"Number of files in the list: {len(file_list)}")
print(file_list)

Now, you should have all the relevant file names in a list, and we can loop over the list using a for loop.

In [ ]:
# Print each file path in the list, one per line
for fp in file_list:
    print(fp)

In [ ]:
# Repeat the analysis steps for each input file in turn:
for fp in file_list:

    # Read selected columns of data, using whitespace (spaces or tabs) as the separator
    # and treating the * characters as NoData values
    data = pd.read_csv(
        fp,
        sep=r'\s+',
        usecols=["YRMODAHRMN","Temperature", "Precipitation","City"],
        na_values=["*", "**", "***", "****", "*****", "******"],
    )

    # Rename the columns
    new_names = {"Temperature": "Temp_Average_F", "Precipitation": "PRECIP", "City": "Location","YRMODAHRMN": "Date"}

    data = data.rename(columns=new_names)

    # Print info about the current input file:
    print(f"Location: {data.at[0, 'Location']}")
    print(f"NUMBER OF OBSERVATIONS: {len(data)}")

    # Create the column that will hold the Celsius values
    col_name = "TEMP_C"
    data[col_name] = None

    # Convert temperatures from Fahrenheit to Celsius
    data["TEMP_C"] = data["Temp_Average_F"].apply(fahr_to_celsius)

    # Convert the timestamp to a string so that we can slice it
    data["Date_STR"] = data["Date"].astype(str)

    # Parse year and month out of the timestamp.
    # Characters 0-3 are the year and characters 4-5 are the month.
    data["MONTH"] = data["Date_STR"].str.slice(start=4, stop=6).astype(int)
    data["YEAR"] = data["Date_STR"].str.slice(start=0, stop=4).astype(int)

    # Extract observations for the month of July
    Julys = data[data["MONTH"] == 7]

    # Take a subset of columns
    Julys = Julys[[ "Temp_Average_F", "TEMP_C", "YEAR", "MONTH"]]

    # Group by year and month
    grouped = Julys.groupby(by=["YEAR", "MONTH"])

    # Get mean values for each group
    monthly_mean = grouped.mean()

    # Print the five warmest Julys for this station
    print(monthly_mean.sort_values(by="TEMP_C", ascending=False).head(5))
    print("\n")

## Summary

This lesson brought together several of the most useful pandas techniques for real analysis work. You can now:

- **Read awkward text files** with `pd.read_csv()`, using `sep=r'\s+'` for varying amounts of whitespace and `na_values` to mark NoData characters.
- **Rename columns** with `.rename(columns=...)`, using a **dictionary** of `old_name: new_name` pairs.
- **Apply your own function** to a whole column - either row by row with `.iterrows()` (easier to follow) or all at once with `.apply()` (shorter and faster).
- **Slice strings** with `.str.slice(start=, stop=)` to pull the year and month out of a timestamp.
- **Group and aggregate** with `.groupby()`, then calculate statistics such as `.mean()` for each group.
- **Repeat an entire analysis over many files** by collecting the file paths with `glob.glob()` and looping over them.

| Watch out for | Reminder |
| --- | --- |
| `.iterrows()` | You get a **copy** of the row - write results back with `data.at[idx, col]`. |
| `.apply()` | Pass the function **name**, with no brackets after it. |
| String slicing | The `stop` position is **not** included, exactly like `range()`. |
| `.groupby()` | Grouping on its own calculates nothing - follow it with `.mean()`, `.sum()`, etc. |

This is the point where pandas starts to feel genuinely powerful: a handful of lines can now summarise years of observations across many stations at once.